# 3D NIfTI Reptile Few-Shot Pipeline

This notebook uses `config.py`, `resize.py`, `split.py`, `dataloader-ben2.py`, and `model.py`.

## Section 0: Download And Extract Dataset


In [ ]:
DOWNLOAD_DATA = False
DATA_URL = "https://zenodo.org/records/7013610/files/data.zip?download=1"
ZIP_PATH = "data.zip"

if DOWNLOAD_DATA:
    !curl -L "{DATA_URL}" -o {ZIP_PATH}
    !unzip -q -o {ZIP_PATH}
else:
    print("Skipping dataset download. Set DOWNLOAD_DATA = True to download and extract data.zip.")


## Section 1: Read Config

In [ ]:
import config

print("Data directory:", config.DATA_DIR)
print("Resized data directory:", config.RESIZED_DATA_DIR)
print("Resize dimensions:", config.RESIZE_DIMS)
print("Batch size:", config.BATCH_SIZE)
print("UNet features:", config.UNET_FEATURES)
print("Reptile outer steps:", config.REPTILE_OUTER_STEPS)
print("Reptile inner steps:", config.REPTILE_INNER_STEPS)
print("Reptile inner LR:", config.REPTILE_INNER_LR)
print("Reptile outer LR:", config.REPTILE_OUTER_LR)

## Section 2: Initialise Functions

In [ ]:
from pathlib import Path
import random

import numpy as np
import torch

from model import UNet3D, bce_dice_loss, dice_score, prepare_episode, validate, test, get_device
import importlib.util

dataloader_path = Path("dataloader-ben2.py")
spec = importlib.util.spec_from_file_location("dataloader_ben2", dataloader_path)
dataloader = importlib.util.module_from_spec(spec)
spec.loader.exec_module(dataloader)
build_3d_dataloader = dataloader.build_3d_dataloader
build_episode_loader = dataloader.build_episode_loader


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)
device = get_device()
print("Device:", device)

## Section 3: Resize Dataset And Split

In [ ]:
RUN_RESIZE = False

if RUN_RESIZE:
    !python3 resize.py --input-dir {config.DATA_DIR} --output-dir {config.RESIZED_DATA_DIR} --x {config.RESIZE_DIMS[0]} --y {config.RESIZE_DIMS[1]} --depth {config.RESIZE_DIMS[2]}
else:
    print("Skipping resize. Set RUN_RESIZE = True to regenerate data-resize/.")

In [ ]:
RUN_SPLIT = True

if RUN_SPLIT:
    !python3 split.py --data-dir {config.RESIZED_DATA_DIR} --output-dir . --train-ratio 0.70 --val-ratio 0.15 --test-ratio 0.15 --seed 42
else:
    print("Skipping split generation.")

## Section 4: Reptile

In [ ]:
model = UNet3D().to(device)
print(model.__class__.__name__)

In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    !python3 train_reptile.py --data-dir {config.RESIZED_DATA_DIR} --train-split train.txt --val-split val.txt --test-split test.txt --n-support 1 --n-query 1 --outer-steps {config.REPTILE_OUTER_STEPS} --inner-steps {config.REPTILE_INNER_STEPS} --inner-lr {config.REPTILE_INNER_LR} --outer-lr {config.REPTILE_OUTER_LR} --val-interval {config.REPTILE_VAL_INTERVAL} --batch-size {config.BATCH_SIZE} --model-path reptile_3d_unet.pth --history-path reptile_history.json --metrics-path reptile_metrics.json
else:
    print("Skipping training. Set RUN_TRAINING = True to call train_reptile.py.")

In [ ]:
import json

model_path = Path("reptile_3d_unet.pth")
history_path = Path("reptile_history.json")
metrics_path = Path("reptile_metrics.json")

history = {"support_loss": [], "query_loss": [], "val_dice": []}
if history_path.exists():
    history = json.loads(history_path.read_text())

if model_path.exists():
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
else:
    print("No saved Reptile model found yet.")

if metrics_path.exists():
    saved_metrics = json.loads(metrics_path.read_text())
    val_metrics = saved_metrics.get("validation", {})
    test_metrics = saved_metrics.get("test", {})
else:
    val_loader = build_3d_dataloader(config.RESIZED_DATA_DIR, "val.txt", config.BATCH_SIZE, shuffle=False)
    test_loader = build_3d_dataloader(config.RESIZED_DATA_DIR, "test.txt", config.BATCH_SIZE, shuffle=False)
    val_metrics = validate(model, val_loader, device=device)
    test_metrics = test(model, test_loader, device=device)

print("Validation:", val_metrics)
print("Test:", test_metrics)

## Section 5: Metrics

In [ ]:
import copy
import matplotlib.pyplot as plt
import torch.optim as optim

N_SHOT_LIST = [1, 3, 5]
RUN_N_SHOT_EVAL = False


def adapt_on_support(base_model, support_images, support_masks, steps=config.REPTILE_INNER_STEPS, lr=config.REPTILE_INNER_LR):
    adapted = copy.deepcopy(base_model).to(device)
    adapted.train()
    optimizer = optim.SGD(adapted.parameters(), lr=lr)

    for _ in range(steps):
        optimizer.zero_grad()
        loss = bce_dice_loss(adapted(support_images), support_masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(adapted.parameters(), 1.0)
        optimizer.step()

    adapted.eval()
    return adapted


def evaluate_n_shot(base_model, n_shot, episodes=10, query_size=1):
    scores = []
    episode_loader = build_episode_loader(
        data_dir=config.RESIZED_DATA_DIR,
        split_file="test.txt",
        n_support=n_shot,
        n_query=query_size,
        episodes=episodes,
    )

    for episode in episode_loader:
        support_images, support_masks, query_images, query_masks = prepare_episode(episode, device)
        adapted = adapt_on_support(base_model, support_images, support_masks)

        with torch.no_grad():
            score = dice_score(adapted(query_images), query_masks)
        if torch.isfinite(score):
            scores.append(score.item())

    return scores


In [ ]:
results = {"Reptile": {}}
if RUN_N_SHOT_EVAL:
    for n_shot in N_SHOT_LIST:
        results["Reptile"][n_shot] = evaluate_n_shot(model, n_shot=n_shot, episodes=10)
else:
    print("Skipping 1/3/5-shot evaluation. Set RUN_N_SHOT_EVAL = True to run it.")

print("=" * 70)
print(f"{'Method':<26} {'1-shot':>10} {'3-shot':>10} {'5-shot':>10}")
print("=" * 70)
print("
  [METHODS]")
if all(n in results["Reptile"] and results["Reptile"][n] for n in N_SHOT_LIST):
    means = [f"{np.mean(results['Reptile'][n]):.3f}" for n in N_SHOT_LIST]
    print(f"  {'Reptile':<26} {means[0]:>10} {means[1]:>10} {means[2]:>10}")
else:
    print("  Reptile                   run n-shot evaluation to fill this table")
print("=" * 70)


In [ ]:
if history["support_loss"]:
    fig, ax = plt.subplots(figsize=(14, 5))
    for key, colour in [("support_loss", "steelblue"), ("query_loss", "darkorange")]:
        values = np.array(history[key], dtype=float)
        values = values[np.isfinite(values)]
        if len(values) == 0:
            continue
        window = max(1, len(values) // 100)
        smoothed = np.convolve(values, np.ones(window) / window, mode="valid")
        ax.plot(range(window - 1, len(values)), smoothed, label=key.replace("_", " "), color=colour)

    ax.set_xlabel("Outer step")
    ax.set_ylabel("Loss")
    ax.set_title("Reptile Meta-Training Loss")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No training history yet.")


In [ ]:
if all(n in results["Reptile"] and results["Reptile"][n] for n in N_SHOT_LIST):
    means = [np.mean(results["Reptile"][n]) for n in N_SHOT_LIST]
    stds = [np.std(results["Reptile"][n]) for n in N_SHOT_LIST]

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar([f"{n}-shot" for n in N_SHOT_LIST], means, yerr=stds, color="steelblue", capsize=5, alpha=0.85)
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01, f"{mean:.3f}", ha="center")
    ax.set_ylabel("Dice Score")
    ax.set_ylim(0, 1.0)
    ax.set_title("Reptile Few-Shot Test Performance")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig("methods_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No n-shot results yet.")


In [ ]:
def visualise_reptile_episode(base_model, n_shot=5, n_examples=3):
    episode = next(build_episode_loader(
        data_dir=config.RESIZED_DATA_DIR,
        split_file="test.txt",
        n_support=n_shot,
        n_query=n_examples,
        episodes=1,
    ))
    support_images, support_masks, query_images, query_masks = prepare_episode(episode, device)
    adapted = adapt_on_support(base_model, support_images, support_masks)

    with torch.no_grad():
        preds = (torch.sigmoid(adapted(query_images)) > 0.5).float().cpu()

    query_images = query_images.cpu()
    query_masks = query_masks.cpu()
    num_examples = min(n_examples, query_images.shape[0])

    fig, axes = plt.subplots(num_examples, 3, figsize=(9, 3 * num_examples))
    if num_examples == 1:
        axes = axes[np.newaxis, :]

    for row in range(num_examples):
        gt_volume = query_masks[row, 0]
        slice_scores = gt_volume.sum(dim=(1, 2))
        slice_idx = int(torch.argmax(slice_scores)) if torch.max(slice_scores).item() > 0 else gt_volume.shape[0] // 2

        image_slice = query_images[row, 0, slice_idx]
        mask_slice = query_masks[row, 0, slice_idx]
        pred_slice = preds[row, 0, slice_idx]

        for ax, data, title in zip(axes[row], [image_slice, mask_slice, pred_slice], ["Input", "Ground Truth", "Reptile Prediction"]):
            ax.imshow(data, cmap="gray")
            ax.set_title(title)
            ax.axis("off")

    plt.tight_layout()
    plt.savefig(f"qualitative_reptile_{n_shot}shot.png", dpi=150, bbox_inches="tight")
    plt.show()


RUN_VISUALISE = False
if RUN_VISUALISE:
    visualise_reptile_episode(model, n_shot=5, n_examples=3)
else:
    print("Skipping qualitative visualisation. Set RUN_VISUALISE = True to run it.")


In [ ]:
print("Final validation metrics:", val_metrics)
print("Final test metrics:", test_metrics)
